# Shown Space Scoring Path Visuals

This notebook uses Shown Space's public game API to recreate field-path visuals from coordinates, then summarizes a team's scoring possessions as an interactive average path and heatmap.

Default team: `glory`.

In [11]:
import sys
sys.path.insert(0, "../src")

import pandas as pd

from ufa import (
    average_scoring_path,
    build_scoring_possessions,
    cluster_scoring_possessions,
    create_scoring_possession_browser,
    create_team_scoring_possession_browser,
    create_team_playstyle_report_browser,
    fetch_shownspace_games,
    fetch_shownspace_season_throws,
    fetch_shownspace_throws_for_games,
    plot_average_scoring_path,
    plot_possession_path,
    plot_representative_paths,
    plot_scoring_heatmap,
    plot_team_representative_path_grid,
    select_representative_paths,
    select_top_paths,
    summarize_path_clusters,
    summarize_team_playstyle,
    summarize_team_playstyles,
)

## Settings

Use `MAX_GAMES = 3` first to validate the visual quickly. Set `MAX_GAMES = None` for the full team season.

In [12]:
SEASON = 2026
TEAM_ID = "breeze"
MAX_GAMES = 3
SAMPLE_GAMES_RANDOMLY = True
RANDOM_STATE = 7
UNIQUE_REPRESENTATIVE_GAMES = True
PULL_RECEIVE_SCORES_ONLY = True
LONG_FIELD_ONLY = True
MAX_START_Y = 45
MIN_FIELD_PROGRESS = 50
EXCLUDE_HUCKS_FROM_TOP_PATHS = True

all_games = fetch_shownspace_games(season=SEASON, final_only=True)
team_games = all_games[
    all_games["HomeTeamID"].str.lower().eq(TEAM_ID.lower())
    | all_games["AwayTeamID"].str.lower().eq(TEAM_ID.lower())
].reset_index(drop=True)

if MAX_GAMES is None:
    games = team_games.copy()
elif SAMPLE_GAMES_RANDOMLY:
    games = (
        team_games
        .sample(n=min(MAX_GAMES, len(team_games)), random_state=RANDOM_STATE)
        .sort_values("StartTimestamp")
        .reset_index(drop=True)
    )
else:
    games = team_games.head(MAX_GAMES).copy()

throws = fetch_shownspace_throws_for_games(games["GameID"].tolist(), delay=0.15)

games[["GameID", "AwayTeamID", "HomeTeamID", "AwayScore", "HomeScore", "Status", "StartTimestamp"]]


,GameID,AwayTeamID,HomeTeamID,AwayScore,HomeScore,Status,StartTimestamp
0,2026-05-10-DC-NY,breeze,empire,26,25,Final,2026-05-10 13:00:00
1,2026-06-13-BOS-DC,glory,breeze,27,18,Final,2026-06-13 19:00:00
2,2026-07-11-PHI-DC,phoenix,breeze,17,28,Final,2026-07-11 19:00:00


In [13]:
possessions, paths = build_scoring_possessions(throws, team_id=TEAM_ID)

print(f"Throws loaded: {len(throws):,}")
print(f"Scoring possessions found for {TEAM_ID}: {len(possessions):,}")

possessions.sort_values("risk_adjusted_aec_per_throw", ascending=False).head(10)

Throws loaded: 1,728
Scoring possessions found for breeze: 72


,possession_id,GameID,team_id,start_timestamp,game_quarter,quarter_point,possession_num,is_home_team,line_type,outcome,...,mean_cp,risk_adjusted_aec_per_throw,total_yards,yards_per_throw,total_throw_distance,avg_throw_distance,max_throw_distance,huck_count,reset_count,lateral_yards
25,2026-05-10-DC-NY|5|7|2|False,2026-05-10-DC-NY,breeze,2026-05-10 13:00:00,5,7,2,False,d_line,goal,...,0.950510,0.950510,15.59,15.590000,15.725511,15.725511,15.725511,0,0,2.06
45,2026-07-11-PHI-DC|1|3|2|True,2026-07-11-PHI-DC,breeze,2026-07-11 19:00:00,1,3,2,True,d_line,goal,...,0.933276,0.933276,17.50,17.500000,18.953849,18.953849,18.953849,0,0,7.28
50,2026-07-11-PHI-DC|1|11|4|True,2026-07-11-PHI-DC,breeze,2026-07-11 19:00:00,1,11,4,True,d_line,goal,...,0.544389,0.544389,71.87,71.870000,74.649376,74.649376,74.649376,1,0,20.18
66,2026-07-11-PHI-DC|4|2|2|True,2026-07-11-PHI-DC,breeze,2026-07-11 19:00:00,4,2,2,True,d_line,goal,...,0.938994,0.471844,37.21,18.605000,37.494460,18.747230,19.361327,0,0,3.92
67,2026-07-11-PHI-DC|4|3|6|True,2026-07-11-PHI-DC,breeze,2026-07-11 19:00:00,4,3,6,True,d_line,goal,...,0.940377,0.469968,31.03,15.515000,31.650297,15.825148,20.245960,0,0,5.97
4,2026-05-10-DC-NY|2|4|3|False,2026-05-10-DC-NY,breeze,2026-05-10 13:00:00,2,4,3,False,o_line,goal,...,0.971753,0.323228,18.33,6.110000,26.904403,8.968134,9.042748,0,0,17.57
61,2026-07-11-PHI-DC|3|7|3|True,2026-07-11-PHI-DC,breeze,2026-07-11 19:00:00,3,7,3,True,d_line,goal,...,0.920116,0.306940,28.76,9.586667,65.267228,21.755743,31.650722,0,0,52.44
63,2026-07-11-PHI-DC|3|10|2|True,2026-07-11-PHI-DC,breeze,2026-07-11 19:00:00,3,10,2,True,d_line,goal,...,0.952312,0.305231,4.21,2.105000,33.095055,16.547528,17.137086,0,1,17.64
1,2026-05-10-DC-NY|1|3|3|False,2026-05-10-DC-NY,breeze,2026-05-10 13:00:00,1,3,3,False,o_line,goal,...,0.959964,0.240262,54.85,13.712500,55.799659,13.949915,23.227839,0,0,8.35
51,2026-07-11-PHI-DC|1|12|2|True,2026-07-11-PHI-DC,breeze,2026-07-11 19:00:00,1,12,2,True,d_line,goal,...,0.871380,0.231497,91.03,22.757500,115.294885,28.823721,57.058676,1,0,57.86


## Long-Field Possession Filter

Use this to focus the visuals on possessions that start farther from the scoring end zone, instead of short-field scores after turnovers.

In [14]:
analysis_possessions = possessions.copy()

if PULL_RECEIVE_SCORES_ONLY:
    analysis_possessions = analysis_possessions[
        analysis_possessions["possession_num"].eq(1)
    ].copy()

if LONG_FIELD_ONLY:
    analysis_possessions = analysis_possessions[
        analysis_possessions["start_y"].le(MAX_START_Y)
        & analysis_possessions["field_progress"].ge(MIN_FIELD_PROGRESS)
    ].copy()

analysis_ids = set(analysis_possessions["possession_id"])
analysis_paths = [
    path for path in paths
    if path["possession_id"].iloc[0] in analysis_ids
]

print(f"All scoring possessions: {len(possessions):,}")
initial_scoring_holds = possessions["possession_num"].eq(1).sum()
print(f"Initial-possession scoring holds: {initial_scoring_holds:,}")
print(f"Analysis possessions: {len(analysis_possessions):,}")

analysis_possessions[[
    "possession_id", "GameID", "possession_num", "start_y", "end_y",
    "field_progress", "throw_count", "total_aec", "aec_per_throw"
]].head(10)


All scoring possessions: 72
Initial-possession scoring holds: 45
Analysis possessions: 44


,possession_id,GameID,possession_num,start_y,end_y,field_progress,throw_count,total_aec,aec_per_throw
0,2026-05-10-DC-NY|1|1|1|False,2026-05-10-DC-NY,1,11.87,107.16,95.29,9,1.089516,0.121057
2,2026-05-10-DC-NY|1|5|1|False,2026-05-10-DC-NY,1,8.59,114.24,105.65,11,1.030926,0.093721
3,2026-05-10-DC-NY|1|7|1|False,2026-05-10-DC-NY,1,4.29,111.07,106.78,15,1.107679,0.073845
5,2026-05-10-DC-NY|2|7|1|False,2026-05-10-DC-NY,1,11.94,112.37,100.43,4,1.001092,0.250273
7,2026-05-10-DC-NY|2|11|1|False,2026-05-10-DC-NY,1,10.91,106.38,95.47,11,1.015889,0.092354
8,2026-05-10-DC-NY|2|13|1|False,2026-05-10-DC-NY,1,11.25,110.32,99.07,6,1.070848,0.178475
10,2026-05-10-DC-NY|3|2|1|False,2026-05-10-DC-NY,1,40.00,106.95,66.95,6,1.048643,0.174774
11,2026-05-10-DC-NY|3|4|1|False,2026-05-10-DC-NY,1,7.89,105.58,97.69,10,0.986872,0.098687
13,2026-05-10-DC-NY|3|8|1|False,2026-05-10-DC-NY,1,1.02,112.58,111.56,7,1.000883,0.142983
18,2026-05-10-DC-NY|4|3|1|False,2026-05-10-DC-NY,1,2.26,106.27,104.01,4,0.721058,0.180264


## Glory Scoring Possession Browser

This is the Shown Space-style possession browser. It uses a custom HTML/SVG field instead of Plotly so the field has simple lines, dots, hover tooltips, and no chart toolbar.

In [15]:
BROWSER_SEASON = 2026
BROWSER_TEAM_ID = "glory"
BROWSER_MAX_GAMES = None  # None means every final game for the team

# Defaults to all Glory scoring possessions. Turn these on only if you want a narrower view.
BROWSER_PULL_RECEIVE_SCORES_ONLY = False
BROWSER_LONG_FIELD_ONLY = False
BROWSER_MAX_START_Y = 45
BROWSER_MIN_FIELD_PROGRESS = 50
BROWSER_EXCLUDE_HUCKS = False

browser_all_games = fetch_shownspace_games(season=BROWSER_SEASON, final_only=True)
browser_games = browser_all_games[
    browser_all_games["HomeTeamID"].str.lower().eq(BROWSER_TEAM_ID.lower())
    | browser_all_games["AwayTeamID"].str.lower().eq(BROWSER_TEAM_ID.lower())
].sort_values("StartTimestamp").reset_index(drop=True)

if BROWSER_MAX_GAMES is not None:
    browser_games = browser_games.head(BROWSER_MAX_GAMES).copy()

browser_throws = fetch_shownspace_throws_for_games(
    browser_games["GameID"].tolist(),
    delay=0.15,
)
browser_possessions, browser_paths = build_scoring_possessions(
    browser_throws,
    team_id=BROWSER_TEAM_ID,
)

if BROWSER_PULL_RECEIVE_SCORES_ONLY:
    browser_possessions = browser_possessions[
        browser_possessions["possession_num"].eq(1)
    ].copy()

if BROWSER_LONG_FIELD_ONLY:
    browser_possessions = browser_possessions[
        browser_possessions["start_y"].le(BROWSER_MAX_START_Y)
        & browser_possessions["field_progress"].ge(BROWSER_MIN_FIELD_PROGRESS)
    ].copy()

if BROWSER_EXCLUDE_HUCKS:
    browser_possessions = browser_possessions[
        browser_possessions["huck_count"].fillna(0).eq(0)
    ].copy()

browser_ids = set(browser_possessions["possession_id"])
browser_paths = [
    path for path in browser_paths
    if path["possession_id"].iloc[0] in browser_ids
]

print(f"Games loaded: {len(browser_games):,}")
print(f"Throws loaded: {len(browser_throws):,}")
print(f"Browser possessions: {len(browser_possessions):,}")

browser_possessions[[
    "possession_id", "GameID", "game_quarter", "quarter_point",
    "possession_num", "throw_count", "start_y", "end_y",
    "field_progress", "total_aec", "aec_per_throw"
]].head(10)


Games loaded: 12
Throws loaded: 6,587
Browser possessions: 261


,possession_id,GameID,game_quarter,quarter_point,possession_num,throw_count,start_y,end_y,field_progress,total_aec,aec_per_throw
0,2026-04-25-DC-BOS|1|1|2|True,2026-04-25-DC-BOS,1,1,2,2,75.03,102.12,27.09,1.010065,0.505033
1,2026-04-25-DC-BOS|1|3|1|True,2026-04-25-DC-BOS,1,3,1,10,14.77,106.96,92.19,0.987499,0.098750
2,2026-04-25-DC-BOS|1|4|2|True,2026-04-25-DC-BOS,1,4,2,10,34.12,115.29,81.17,1.145452,0.114545
3,2026-04-25-DC-BOS|1|5|2|True,2026-04-25-DC-BOS,1,5,2,20,19.74,104.06,84.32,0.891622,0.044581
4,2026-04-25-DC-BOS|1|6|2|True,2026-04-25-DC-BOS,1,6,2,7,49.80,108.32,58.52,1.004663,0.143523
5,2026-04-25-DC-BOS|1|8|1|True,2026-04-25-DC-BOS,1,8,1,9,46.70,115.16,68.46,1.000726,0.111192
6,2026-04-25-DC-BOS|2|1|1|True,2026-04-25-DC-BOS,2,1,1,15,8.45,105.29,96.84,1.041824,0.069455
7,2026-04-25-DC-BOS|2|2|2|True,2026-04-25-DC-BOS,2,2,2,3,89.80,105.61,15.81,1.006915,0.335638
8,2026-04-25-DC-BOS|2|4|1|True,2026-04-25-DC-BOS,2,4,1,10,13.80,114.32,100.52,0.984443,0.098444
9,2026-04-25-DC-BOS|2|6|1|True,2026-04-25-DC-BOS,2,6,1,4,13.87,107.54,93.67,1.000987,0.250247


## Widget Display Check

Run this small check if the browser output looks blank. If the slider does not appear, restart the kernel and rerun the import cell.

In [16]:
import ipywidgets as widgets
widgets.IntSlider(description="widget test")


IntSlider(value=0, description='widget test')

In [17]:
import sys
sys.path.insert(0, "../src")

import importlib
import ufa.shownspace_paths as shownspace_paths
importlib.reload(shownspace_paths)

create_scoring_possession_browser = shownspace_paths.create_scoring_possession_browser
create_team_scoring_possession_browser = shownspace_paths.create_team_scoring_possession_browser

In [18]:
from IPython.display import display

team_browser = create_team_scoring_possession_browser(
    season=BROWSER_SEASON,
    default_team_id=BROWSER_TEAM_ID,
    final_only=True,
    max_games=BROWSER_MAX_GAMES,
    pull_receive_scores_only=BROWSER_PULL_RECEIVE_SCORES_ONLY,
    long_field_only=BROWSER_LONG_FIELD_ONLY,
    max_start_y=BROWSER_MAX_START_Y,
    min_field_progress=BROWSER_MIN_FIELD_PROGRESS,
    exclude_hucks=BROWSER_EXCLUDE_HUCKS,
    n_shape_clusters=8,
)

display(team_browser)


## Team Playstyle Summary Report

These summaries use the same scoring-possession shape features as the browser. The text is rule-based, so every phrase traces back to the metrics shown in the table.


In [19]:
PLAYSTYLE_COLUMNS = [
    "team_id",
    "possessions",
    "primary_shapes",
    "attack_spaces",
    "pace_style",
    "field_width_style",
    "huck_usage",
    "reset_usage",
    "efficiency_note",
    "playstyle_summary",
]

BACKING_METRIC_COLUMNS = [
    "avg_throws",
    "avg_width",
    "avg_side_switches",
    "avg_directness",
    "avg_middle_usage",
    "avg_sideline_usage",
    "avg_hucks",
    "avg_resets",
    "avg_aec_per_throw",
    "avg_cp",
]

team_playstyle = summarize_team_playstyle(
    browser_possessions,
    browser_paths,
    team_id=BROWSER_TEAM_ID,
    n_shape_clusters=8,
)

team_playstyle_table = pd.DataFrame([team_playstyle])

display(
    create_team_playstyle_report_browser(
        team_playstyle,
        team_playstyle_table,
        title=f"{BROWSER_TEAM_ID.title()} playstyle summary, {BROWSER_SEASON}",
    )
)


HTML(value='\n    <div class="ufa-playstyle-browser">\n      <style>\n        .ufa-playstyle-browser {\n      …

In [20]:
PLAYSTYLE_TEAM_IDS = ["glory", "empire", "spiders"]
PLAYSTYLE_MAX_GAMES = None

playstyle_rows = []
for playstyle_team_id in PLAYSTYLE_TEAM_IDS:
    team_games = browser_all_games[
        browser_all_games["HomeTeamID"].str.lower().eq(playstyle_team_id.lower())
        | browser_all_games["AwayTeamID"].str.lower().eq(playstyle_team_id.lower())
    ].sort_values("StartTimestamp").reset_index(drop=True)

    if PLAYSTYLE_MAX_GAMES is not None:
        team_games = team_games.head(PLAYSTYLE_MAX_GAMES).copy()

    team_throws = fetch_shownspace_throws_for_games(
        team_games["GameID"].tolist(),
        delay=0.15,
    )
    team_possessions, team_paths = build_scoring_possessions(
        team_throws,
        team_id=playstyle_team_id,
    )
    playstyle_rows.append(
        summarize_team_playstyle(
            team_possessions,
            team_paths,
            team_id=playstyle_team_id,
            n_shape_clusters=8,
        )
    )

team_playstyle_table = pd.DataFrame(playstyle_rows)
selected_team_row = team_playstyle_table[
    team_playstyle_table["team_id"].astype(str).str.lower().eq(BROWSER_TEAM_ID.lower())
]
if selected_team_row.empty:
    selected_team_playstyle = team_playstyle_table.iloc[0]
else:
    selected_team_playstyle = selected_team_row.iloc[0]

display(
    create_team_playstyle_report_browser(
        selected_team_playstyle,
        team_playstyle_table,
        title=f"Team playstyle comparison, {BROWSER_SEASON}",
    )
)


KeyboardInterrupt: 

## Average Scoring Path

The average path is progress-normalized. Each scoring possession is resampled to fixed progress checkpoints from possession start to goal, then the checkpoint coordinates are averaged.

In [ ]:
avg_path = average_scoring_path(paths)
avg_path

In [ ]:
fig = plot_average_scoring_path(
    avg_path,
    paths=paths,
    title=f"{TEAM_ID.title()} average scoring path, {SEASON} sample",
    show_individual_paths=True,
)
fig.show()

## Real Representative Paths

The mean path is useful as a center-of-gravity check, but it can hide the actual bends, resets, hucks, and lateral movement that make an offense interesting. These cells keep real possessions intact and then pick examples worth studying.

In [ ]:
clustered_possessions = cluster_scoring_possessions(analysis_possessions, analysis_paths, n_clusters=4)
cluster_summary = summarize_path_clusters(clustered_possessions)
cluster_summary

In [ ]:
# Try to show each representative path from a different game when possible.
representative_paths = select_representative_paths(
    clustered_possessions,
    analysis_paths,
    group_column="path_cluster",
    unique_games=UNIQUE_REPRESENTATIVE_GAMES,
)

rep_fig = plot_representative_paths(
    representative_paths,
    title=f"{TEAM_ID.title()} representative scoring path styles, {SEASON} sample",
)
rep_fig.show()

In [ ]:
top_path_possessions = clustered_possessions.copy()
top_path_source_paths = analysis_paths

if EXCLUDE_HUCKS_FROM_TOP_PATHS:
    top_path_possessions = top_path_possessions[
        top_path_possessions["huck_count"].fillna(0).eq(0)
    ].copy()
    top_path_ids = set(top_path_possessions["possession_id"])
    top_path_source_paths = [
        path for path in analysis_paths
        if path["possession_id"].iloc[0] in top_path_ids
    ]

top_paths = select_top_paths(
    top_path_possessions,
    top_path_source_paths,
    metric="aec_per_throw",
    n=3,
)

top_path_title = "highest non-huck long-field aEC per throw" if EXCLUDE_HUCKS_FROM_TOP_PATHS else "highest long-field aEC per throw"

best_fig = plot_possession_path(
    top_paths[0],
    title=f"{TEAM_ID.title()} {top_path_title} scoring possession, {SEASON} sample",
)
best_fig.show()


## Middle Non-Huck Scoring Possessions

The highest `aEC_per_throw` possession can still be an outlier. This view sorts the filtered non-huck possessions by `aEC_per_throw` and plots the middle five, which should be closer to normal efficient offense.

In [ ]:
MIDDLE_PATH_COUNT = 5
MIDDLE_PATH_METRIC = "aec_per_throw"

middle_source = top_path_possessions.sort_values(MIDDLE_PATH_METRIC).reset_index(drop=True)
middle_count = min(MIDDLE_PATH_COUNT, len(middle_source))
middle_start = max((len(middle_source) - middle_count) // 2, 0)
middle_path_possessions = middle_source.iloc[
    middle_start:middle_start + middle_count
].copy()

middle_path_lookup = {
    path["possession_id"].iloc[0]: path
    for path in top_path_source_paths
}
middle_paths = {
    f"middle {rank + 1}: {row[MIDDLE_PATH_METRIC]:.3f}": middle_path_lookup[row["possession_id"]]
    for rank, (_, row) in enumerate(middle_path_possessions.iterrows())
    if row["possession_id"] in middle_path_lookup
}

middle_path_possessions[[
    "possession_id", "GameID", "start_y", "end_y", "field_progress",
    "throw_count", "huck_count", MIDDLE_PATH_METRIC
]]


In [ ]:
middle_fig = plot_representative_paths(
    middle_paths,
    title=f"{TEAM_ID.title()} middle {len(middle_paths)} non-huck long-field scoring possessions, {SEASON} sample",
)
middle_fig.show()


## Compare Teams Side By Side

The readable comparison is one possession style at a time. Build representative paths for each team, then choose a style like `huck`, `reset`, `quick`, or `methodical` to compare across teams.

In [ ]:
TEAM_IDS_TO_COMPARE = ["glory", "empire", "spiders"]

team_representative_paths = {}
team_cluster_summaries = {}

for compare_team_id in TEAM_IDS_TO_COMPARE:
    compare_games = all_games[
        all_games["HomeTeamID"].str.lower().eq(compare_team_id.lower())
        | all_games["AwayTeamID"].str.lower().eq(compare_team_id.lower())
    ].reset_index(drop=True)

    if MAX_GAMES is None:
        selected_games = compare_games.copy()
    elif SAMPLE_GAMES_RANDOMLY:
        selected_games = (
            compare_games
            .sample(n=min(MAX_GAMES, len(compare_games)), random_state=RANDOM_STATE)
            .sort_values("StartTimestamp")
            .reset_index(drop=True)
        )
    else:
        selected_games = compare_games.head(MAX_GAMES).copy()

    compare_throws = fetch_shownspace_throws_for_games(
        selected_games["GameID"].tolist(),
        delay=0.15,
    )
    compare_possessions, compare_paths = build_scoring_possessions(
        compare_throws,
        team_id=compare_team_id,
    )

    compare_analysis_possessions = compare_possessions.copy()
    if PULL_RECEIVE_SCORES_ONLY:
        compare_analysis_possessions = compare_analysis_possessions[
            compare_analysis_possessions["possession_num"].eq(1)
        ].copy()
    if LONG_FIELD_ONLY:
        compare_analysis_possessions = compare_analysis_possessions[
            compare_analysis_possessions["start_y"].le(MAX_START_Y)
            & compare_analysis_possessions["field_progress"].ge(MIN_FIELD_PROGRESS)
        ].copy()

    compare_ids = set(compare_analysis_possessions["possession_id"])
    compare_analysis_paths = [
        path for path in compare_paths
        if path["possession_id"].iloc[0] in compare_ids
    ]

    compare_clustered = cluster_scoring_possessions(
        compare_analysis_possessions,
        compare_analysis_paths,
        n_clusters=4,
    )
    team_cluster_summaries[compare_team_id] = summarize_path_clusters(compare_clustered)
    team_representative_paths[compare_team_id] = select_representative_paths(
        compare_clustered,
        compare_analysis_paths,
        group_column="path_cluster",
        unique_games=UNIQUE_REPRESENTATIVE_GAMES,
    )

comparison_counts = pd.DataFrame([
    {
        "team_id": team_id,
        "representative_paths": len(representative_paths),
    }
    for team_id, representative_paths in team_representative_paths.items()
])
comparison_counts


In [ ]:
STYLE_TO_COMPARE = "huck"  # Try "reset", "quick", "methodical", or None

style_title = (
    "All styles"
    if STYLE_TO_COMPARE is None
    else STYLE_TO_COMPARE.title()
)

comparison_fig = plot_team_representative_path_grid(
    team_representative_paths,
    title=f"{style_title} representative scoring paths by team, {SEASON}",
    style_filter=STYLE_TO_COMPARE,
    show_arrows=False,
)
comparison_fig.show()


### Optional: All Styles

This is busier, but useful as a quick overview after the one-style comparison makes sense.

In [ ]:
all_styles_fig = plot_team_representative_path_grid(
    team_representative_paths,
    title=f"All representative scoring path styles by team, {SEASON}",
    style_filter=None,
    show_arrows=False,
)
all_styles_fig.show()


## Catch Location Heatmap

This shows where completed throws in scoring possessions are caught.

In [ ]:
heatmap = plot_scoring_heatmap(
    analysis_paths,
    title=f"{TEAM_ID.title()} scoring-possession catch heatmap, {SEASON} sample",
)
heatmap.show()

## Full Team Season

After the sample plots look right, run the full team season by setting `MAX_GAMES = None` below.

In [ ]:
MAX_GAMES = None

games_full, throws_full = fetch_shownspace_season_throws(
    season=SEASON,
    team_id=TEAM_ID,
    max_games=MAX_GAMES,
    delay=0.15,
)

possessions_full, paths_full = build_scoring_possessions(throws_full, team_id=TEAM_ID)
avg_path_full = average_scoring_path(paths_full)

print(f"Games loaded: {len(games_full):,}")
print(f"Throws loaded: {len(throws_full):,}")
print(f"Scoring possessions found for {TEAM_ID}: {len(possessions_full):,}")

possessions_full.sort_values("risk_adjusted_aec_per_throw", ascending=False).head(20)

In [ ]:
fig_full = plot_average_scoring_path(
    avg_path_full,
    paths=paths_full,
    title=f"{TEAM_ID.title()} average scoring path, {SEASON}",
    show_individual_paths=True,
)
fig_full.show()

In [ ]:
heatmap_full = plot_scoring_heatmap(
    paths_full,
    title=f"{TEAM_ID.title()} scoring-possession catch heatmap, {SEASON}",
)
heatmap_full.show()